In [59]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss

In [60]:
df = pd.read_csv('../0.dataset/dataset_serangan_jantung_clean.csv')
df_x = df.drop(columns='Serangan_Jantung')
df_y = df['Serangan_Jantung']

# 1.Binary Classification K-Nearest Neighbors

In [61]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [62]:
params = {
 'n_neighbors' : list(range(1, 6)),
 'weights': ['uniform', 'distance'],
 'metric': ['euclidean', 'manhattan', 'minkowski'],
 'p': [1,2]
}

knn = KNeighborsClassifier()
random_search = RandomizedSearchCV(estimator=knn,param_distributions=params,
                                   n_iter=30,cv=6,scoring='accuracy',random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
best_model_knn = random_search.best_estimator_

sfs_knn = SequentialFeatureSelector(estimator=best_model_knn,n_features_to_select=13,direction='forward')
sfs_knn.fit(X_train,y_train)

X_train_selected = sfs_knn.transform(X_train)
X_test_selected = sfs_knn.transform(X_test)

fitur_terpilih = X_train.columns[sfs_knn.get_support()]
best_model_knn.fit(X_train_selected,y_train)

print(f'\nHyperparameter Terbaik: {random_search.best_params_}')
print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')


Hyperparameter Terbaik: {'weights': 'distance', 'p': 1, 'n_neighbors': 5, 'metric': 'minkowski'}

Akurasi Validasi Terbaik: 72.83%

Fitur Terbaik Yang Terpilih:
['Usia', 'Jenis_Kelamin', 'Tipe_Nyeri_Dada', 'Tekanan_Darah_Rest', 'Kolesterol', 'Gula_Darah_Puasa', 'Detak_Jantung_Max', 'Angina_Olahraga', 'Oldpeak_ST', 'Kemiringan_ST', 'Jumlah_Pembuluh_Darah', 'Thalassemia', 'BMI']


In [63]:
test_accuracy = best_model_knn.score(X_test_selected,y_test)

y_pred_test = best_model_knn.predict(X_test_selected)
y_prob_test = best_model_knn.predict_proba(X_test_selected)

y_pred_train = best_model_knn.predict(X_train_selected)
y_prob_train = best_model_knn.predict_proba(X_train_selected)

print(f'\nAkurasi Data Test: {test_accuracy*100:.2f}%')



Akurasi Data Test: 71.33%


In [64]:
def evaluate_model(pred,test,prob,evaluate,model_name='K-Nearest Neighbors'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test,pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:,1])
    logloss = log_loss(test,prob)

    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [65]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    accuracy_train = df_eval_train['Accuracy'].values[0]
    accuracy_test = df_eval_test['Accuracy'].values[0]
    gap = accuracy_train - accuracy_test

    if accuracy_train < 0.60 and accuracy_test <0.60:
        status = 'Undeerfitting (Akurasi Rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f})'
    elif gap < -0.05:
        status = 'Overfitting / Data Leakage (Test > Train)'
    else:
        status = 'Good Fit'

    df_combined['Status'] = status
    return df_combined

In [66]:
df_eval_train = evaluate_model(y_pred_train,y_train,y_prob_train,evaluate='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
                 Model Evaluated  Accuracy  Precision   Recall  F1-Score   ROC-AUC      Log Loss                   Status
0  K-Nearest Neighbors  Training  1.000000    1.00000  1.00000  1.000000  1.000000  2.220446e-16  Overfitting (gap:28.67)
1  K-Nearest Neighbors      Test  0.713333    0.73822  0.79661  0.766304  0.775665  1.217236e+00  Overfitting (gap:28.67)


